In [1]:
import wave 
import pyroomacoustics as pa
import numpy as np
import scipy.signal as sp
import scipy

from calc_steering_vector import calculate_steering_vector

In [24]:
# 遅延和アレイ
def execute_doa_sparse_separation(input_vectors, steering_vectors, omega):
    """
    input_vectors: マイクロホン入力信号 (num_microhones, freq_bins, time_frames)
    steering_vectors: (freq_bins, target_signal_range, time_frames)
    omega: 目的音の範囲(num_sources, target_signal_range)
    """
    inner_product = np.einsum("kim,mkt->kit", np.conjugate(steering_vectors), input_vectors)
    """inner_product: (freq_bins, target_signal_range, time_frames)"""
    n_omega = np.shape(omega)[1]
    estimate_doas = np.argmax(np.abs(inner_product), axis=1)
    """estimate_doas: (freq_bins, time_frames)"""
    estimate_doas_mask = np.identity(n_omega)[estimate_doas]
    """estimate_doas_mask: (freq_bins, time_frames, target_signal_range)"""
    output_mask = np.einsum("kti,si->skt", estimate_doas_mask, omega)
    """output_mask: (num_sources, freq_bins, time_frames)"""
    y = np.einsum("skt,mkt->mskt", output_mask, input_vectors)
    """y: (num_microphones, num_sources, freq_bins, time_frames)"""
    return y

In [25]:
# SNRを測る
def calculate_snr(target, out):
    """
    target: 目的音 (num_samples, )
    out: 雑音除去後の信号 (num_samples, )
    """
    wave_length = np.minimum(np.shape(target)[0], np.shape(out)[0])
    # 消し残った雑音
    target = target[:wave_length]
    out = out[:wave_length]
    noise = target - out
    snr = 10. * np.log10(np.sum(np.square(target)) / np.sum(np.square(noise)))
    return snr

In [26]:
def modify_angle_diff(diff):
    diff = np.where(diff < -np.pi, diff + np.pi * 2, diff)
    diff = np.where(diff > np.pi, diff - np.pi * 2, diff)
    return diff

In [27]:
if __name__ == "__main__":
    # 乱数の種を初期化
    np.random.seed(0)
    # 畳み込みに用いる波形
    clean_wave_files = ["./CMU_ARCTIC/cmu_us_aew/wav/arctic_a0001.wav", "./CMU_ARCTIC/cmu_us_axb/wav/arctic_a0002.wav"]
    # 雑音だけの区間のフレーム数
    n_noise_only = 40000
    # 音源数
    n_sources = len(clean_wave_files)
    # 音声波形の長さを調べる
    n_samples = 0
    # ファイルを読み込む
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        if n_samples<wav.getnframes():
            n_samples=wav.getnframes()
        wav.close()
    clean_data = np.zeros([n_sources, n_samples])

    # ファイルを読み込む
    s = 0
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        data = wav.readframes(wav.getnframes())
        data = np.frombuffer(data, dtype=np.int16)
        data = data/np.iinfo(np.int16).max
        clean_data[s, :wav.getnframes()] = data
        wav.close()
        s = s+1

    # シミュレーションのパラメータ
    n_sim_sources = 2
    # サンプリングレート [Hz]
    sample_rate = 16000
    # フレームサイズ
    N = 1024
    # 周波数の数
    Nk = N / 2 + 1
    # 各ビンの周波数
    freqs = np.arange(0, Nk, 1) * sample_rate / N
    # 音声と雑音の比率 [dB]
    SNR = 90.
    # 方位角の閾値
    azimuth_thresh = 30
    # 部屋の大きさ
    room_dim = np.r_[10.0, 10.0, 10.0]
    # マイクロホンアレイを置く部屋の場所
    mic_array_loc = room_dim / 2 + np.random.randn(3) * 0.1
    # マイクロホンアレイのマイクロホン配置
    mic_directions = np.array(
        [[np.pi/2, theta/180 * np.pi] for theta in np.arange(0, 360, 10)]
    )
    distance = 0.02
    mic_alignments = np.zeros((3, mic_directions.shape[0]), dtype=mic_directions.dtype)
    mic_alignments[0, :] = np.cos(mic_directions[:, 1]) * np.sin(mic_directions[:, 0]) 
    mic_alignments[1, :] = np.sin(mic_directions[:, 1]) * np.sin(mic_directions[:, 0]) 
    mic_alignments[2, :] = np.cos(mic_directions[:, 0])
    mic_alignments *= distance

    # マイクロホン数
    n_channels = np.shape(mic_alignments)[0]
    # get the microphone array
    R  = mic_alignments + mic_array_loc[:, None]
    """R: (3D-coordinate(x,y,z)=3, num_microphones)"""
    # 部屋を生成する
    room = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    room_no_noise_left = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    room_no_noise_right = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    # 用いるマイクロホンアレイの情報を設置する
    room.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_left.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_right.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    # 音源の場所
    doas  = np.array(
        [[np.pi/2, np.pi],
        [np.pi/2, 0]]
        )
    # 音源とマイクロホンの距離
    distance = 1.
    source_locations = np.zeros((3, doas.shape[0]), dtype=doas.dtype)
    """source_locations: (xyz, num_sources)"""
    source_locations[0,  :] = np.cos(doas[:, 1]) * np.sin(doas[:, 0]) 
    source_locations[1,  :] = np.sin(doas[:, 1]) * np.sin(doas[:, 0])
    source_locations[2,  :] = np.cos(doas[:, 0])
    source_locations *= distance
    source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置

    # ステアリングベクトルを算出するための仮想的な音源方向
    virtual_doas = np.array(
        [[np.pi/2, theta/180 * np.pi] for theta in np.arange(0, 360, 5)]
    )
    virtual_source_locations = np.zeros((3, virtual_doas.shape[0]), dtype=virtual_doas.dtype)
    """virtual_source_locations: (xyz, num_sources)"""
    virtual_source_locations[0,  :] = np.cos(virtual_doas[:, 1]) * np.sin(virtual_doas[:, 0]) 
    virtual_source_locations[1,  :] = np.sin(virtual_doas[:, 1]) * np.sin(virtual_doas[:, 0])
    virtual_source_locations[2,  :] = np.cos(virtual_doas[:, 0])
    virtual_source_locations *= 100
    virtual_source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置

    # 仮想的な音源方向のステアリングベクトル作成
    virtual_steering_vectors = calculate_steering_vector(R, virtual_source_locations, freqs, is_use_far=True)

    # 所望音の方向から±thresh度以内
    omega = np.array([np.abs(modify_angle_diff(virtual_doas[:, 1] - doas[s, 1])) < azimuth_thresh / 180 * np.pi for s in range(n_sim_sources)]).astype(np.float)

    # 各音源をシミュレーションに追加する
    for s in range(n_sim_sources):
        clean_data[s] /= np.std(clean_data[s])
        room.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 0:
            room_no_noise_left.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 1:
            room_no_noise_right.add_source(source_locations[:, s], signal=clean_data[s])

    # シミュレーションを回す
    room.simulate(snr=SNR)
    room_no_noise_left.simulate(snr=90)
    room_no_noise_right.simulate(snr=90)

    # 畳み込んだ波形を取得する
    multi_conv_data = room.mic_array.signals
    """multi_conv_data: (num_channels, num_samples)"""
    multi_conv_data_left_no_noise = room_no_noise_left.mic_array.signals
    """multi_conv_data_left_no_noise: (num_channels, num_samples)"""
    multi_conv_data_right_no_noise = room_no_noise_right.mic_array.signals
    """multi_conv_data_right_no_noise: (num_channels, num_samples)"""

    # 短時間フーリエ変換
    f, t, stft_data = sp.stft(multi_conv_data, fs=sample_rate, window="hann", nperseg=N)
    """f: (freq_bins,), t: (1,), stft_data:(num_microphones, freq_bins, time_frames)"""

    # DOA情報を使って分離
    y_doa = execute_doa_sparse_separation(stft_data, virtual_steering_vectors, omega)

    # 時間領域の波形に戻す
    t, y_doa_left = sp.istft(y_doa[0, 0, ...], fs=sample_rate, window="hann", nperseg=N)
    t, y_doa_right = sp.istft(y_doa[0, 1, ...], fs=sample_rate, window="hann", nperseg=N)

    # SNRを測る
    snr_pre = calculate_snr(multi_conv_data_left_no_noise[0, ...], multi_conv_data[0, ...]) + calculate_snr(multi_conv_data_right_no_noise[0, ...], multi_conv_data[0, ...])
    snr_doa_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], y_doa_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], y_doa_right)
    snr_pre /= 2
    snr_doa_post /= 2
    print("result ΔSNR [dB]")
    print("ΔSNR:{:.2f}".format(snr_doa_post-snr_pre))

result ΔSNR [dB]
ΔSNR:11.01
